In [1]:
import pyarrow as pa
import pyarrow.dataset as ds
import pyarrow.compute as pc
import pyarrow.parquet as pq
import pyarrow.csv as pv
import pandas as pd


import s3fs
import pandas
import geopandas
from pathlib import Path

import stream_utils.S3hsclient as hsclient

from tqdm.notebook import tqdm


from data_preprocess import parquet_tools, upload_to_hydroshare

Please Enter Your HydroShare Credentials


Username:  TonyCastronova
Password for TonyCastronova:  ········


In [2]:
root_dir = Path('stream-v1-data-original')
pq_dir = Path('stream-v1-data-parquet')
pq_dir.mkdir(exist_ok=True)

resource_id = '4553ef8a14884e4199c7018975b8faaa'

## Create Metadata Parquet

In [51]:
df = pandas.read_csv(root_dir / "01_metadata/metadata.csv")
df.to_parquet(pq_dir / 'metadata.parquet', index=False)

## Create Static Catchment Characteristics Parquet

In [52]:
df = pandas.read_csv(root_dir / "04_static_catchment_characteristics/static_catchment_characteristics.csv")
df.to_parquet(pq_dir / 'static_catchment_characteristics.parquet', index=False)

## Create Watersheds Parquet

In [30]:
# loop through all watershed shapefiles and combine
# them into a single geodataframe.
gdfs = []
shapefiles = list((root_dir / "03_shapefiles").glob('*.shp'))
for shapefile in tqdm(shapefiles, desc="Loading Shapefiles", unit="file"):
    gdfs.append(geopandas.read_file(shapefile))
gdf = pandas.concat(gdfs)

# save the watersheds geodataframe to parquet
gdf.to_parquet(pq_dir / 'watersheds.parquet', index=False)

Loading Shapefiles:   0%|          | 0/4268 [00:00<?, ?file/s]

## Create LULC, Anthropogenic, Grab Sample, and Water Quality Parquets

In [10]:
# def create_parquet(input_data_dir, output_filename,
#                    date_cols = [], sort_by = None):
#     """
#     Loops through all of the CSV files in a given directory, loads them
#     into a Pandas object, then saves the result to Parquet.
#     """
    
#     input_data_dir = Path(input_data_dir)
#     dfs = []

#     # loop through all of the csv files in the current directory
#     # and build a list of Pandas DataFrames. Then concatenate
#     # the list of DataFrames into a single object.
#     for f in input_data_dir.glob('*.csv'):
#         df_temp = pandas.read_csv(f,
#                                  parse_dates=date_cols)
#         df_temp['STREAM_ID'] = f.stem
#         dfs.append(df_temp)
#     df = pandas.concat(dfs)

#     # sort the values in the DataFrame using the sort_by
#     # input argument. This is used to improve query performance
#     if sort_by is not None:
#         df.sort_values(by=sort_by, inplace=True)

#     # save the result as a parquet file in the user-specified
#     # output location.
#     df.to_parquet(output_filename, index=False)
    

In [54]:
# process land use land cover data and dynamic_antropogenic data.
print("Processing land use land cover data...", end="", flush=True)
parquet_tools.create_single_parquet(
    root_dir / "06_dynamic_lulc",
    pq_dir / "dynamic_lulc.parquet",
    date_cols=["DateTime"],
    sort_by=['STREAM_ID', 'DateTime']
)
print("done")


# create_parquet(root_dir / '06_dynamic_lulc',
#                pq_dir / 'dynamic_lulc.parquet',
#                date_cols=['DateTime'],
#                sort_by=['STREAM_ID', 'DateTime'])

Processing land use land cover data...done


In [56]:
print("Processing dynamic anthropogenic data...", end="", flush=True)
parquet_tools.create_single_parquet(
    root_dir / "07_dynamic_anthropogenic",
    pq_dir / "dynamic_antropogenic.parquet",
    date_cols=["DateTime"],
    sort_by=['STREAM_ID', 'DateTime']
)
print("done")

# create_parquet(root_dir / '07_dynamic_anthropogenic',
#                pq_dir / 'dynamic_anthropogenic.parquet',
#                date_cols=['DateTime'],
#                sort_by=['STREAM_ID', 'DateTime'])

Processing dynamic anthropogenic data...done


In [57]:
# process grab sample data. This has a date column named "DateTime" and we will sort
print("Processing grab sample data...", end="", flush=True)
parquet_tools.create_single_parquet(
    root_dir / "09_grab_samples",
    pq_dir / "grab_samples.parquet",
    date_cols=["DateTime"],
    sort_by=['STREAM_ID', 'DateTime']
)
print("done")

# create_parquet(root_dir / '09_grab_samples',
#                pq_dir / 'grab_samples.parquet',
#                date_cols=['DateTime'],
#                sort_by=['STREAM_ID', 'DateTime'])

Processing grab sample data...done


## Create Streamflow Discharge Parquet

In [ ]:
print("Processing streamflow discharge data...", end="", flush=True)
parquet_tools.create_hive_parquet(
    root_dir / "08_streamflow_discharge",
    pq_dir / "streamflow",
    date_cols=["DateTime"],
    partition_by=[
        ("STREAM_ID", pa.large_string()),
    ],
    sort_by="DateTime",
)



Upload to the Hydroshare Resource

In [5]:
upload_to_hydroshare.upload_data(resource_id,
                                str(pq_dir / "streamflow") + '/',
                                target_prefix='streamflow')

Uploading Data:   0%|          | 0/2186 [00:00<?, ?file/s]

2186 uploaded [elapsed time: 2320.23 seconds]


## Create Water Quality Parquet

In [3]:
columns = { 
    'DateTime':'date32',
    'STREAM_ID': 'large_string',
    'WTemp_C':'float32',
    'SpC_uScm':'float32',
    'DO_mgL':'float32',
    'pH':'float32',
    'Turb_FNU':'float32',
    'Turb_NTU':'float32',
    'NO3_mgNL':'float32',
    'PO4_mgPL':'float32',
    'fDOM_QSU':'float32',
    'fDOM_RFU':'float32',
    'Chla_ugL':'float32',
    'Chla_RFU':'float32',
    'PC_RFU':'float32',
}

parquet_tools.create_hive_parquet(
    root_dir / "02_waterquality_timeseries",
    pq_dir / "waterquality_timeseries",
    date_cols=["DateTime"],
    columns=columns,
    partition_by=[
        ("STREAM_ID", pa.large_string()),
    ],
    sort_by="DateTime",
)


# print("Processing water quality discharge data...", end="", flush=True)
# parquet_tools.create_hive_parquet(
#     root_dir / "02_waterquality_timeseries",
#     pq_dir / "waterquality_timeseries",
#     date_cols=["DateTime"],
#     partition_by=[
#         ("STREAM_ID", pa.large_string()),
#     ],
#     sort_by="DateTime",
# )

Processing water quality discharge data...
    Loading data csv data into Pandas...

Loading Data:   0%|          | 0/4254 [00:00<?, ?file/s]

Error processing file: STREAM-gauge-3320
done [elapsed time: 107.43 seconds]
    Converting DataFrame into PyArrow table...done [elapsed time: 6.24 seconds]
    Writing PyArrow table to Parquet files...done [elapsed time: 64.16 seconds]


In [3]:
upload_to_hydroshare.upload_data(resource_id,
                                str(pq_dir / "waterquality_timeseries") + '/',
                                target_prefix='waterquality_timeseries')

SystemExit: 1

/Users/castro/Documents/work/stream/.pixi/envs/default/lib/python3.12/site-packages/IPython/core/interactiveshell.py:3709: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)


## Create Dynamic Historical Meteorology

In [3]:
root_dir = Path('/Volumes/ColdStorage/STREAM')

In [3]:
columns = { 
    'DateTime':'date32',
    'STREAM_ID': 'large_string',
    'APCP_surface':'float32',
    'DLWRF_surface':'float32',
    'DSWRF_surface':'float32',
    'PRES_surface':'float32',
    'SPFH_2maboveground':'float32',
    'TMP_2maboveground':'float32',
    'UGRD_10maboveground':'float32',
    'VGRD_10maboveground':'float32',
}

In [4]:



parquet_tools.create_hive_parquet_from_table(
    root_dir / "05_dynamic_historical_meteorology",
    root_dir / "dynamic_historical_meteorology",
    date_cols=["DateTime"],
    columns=columns,
    partition_by=[
        ("STREAM_ID", pa.large_string()),
    ],
    sort_by="DateTime",
)



    Streaming CSV data into hive-partitioned Parquet...

Processing Data:   0%|          | 0/4268 [00:00<?, ?file/s]

done [elapsed time: 2142.24 seconds]


In [5]:
upload_to_hydroshare.upload_data(resource_id,
                                str(root_dir / "dynamic_historical_meteorology") + '/',
                                target_prefix='dynamic_historical_meteorology',
                                overwrite=False)

Skipping 0 files


Uploading Data:   0%|          | 0/545 [00:00<?, ?file/s]

545 uploaded [elapsed time: 2066.57 seconds]
